# Data skew e a Spark UI

## Task 4: investigar um job travado numa task só (data skew) com a Spark UI

## O novo pedido

Nomes dos tipos de lance **em português**: um join do feed com uma tabela de descrições.

**Data skew**: uma chave concentra boa parte das linhas, e o shuffle manda todas as linhas de uma chave para a mesma
partição. Uma task demora muito, e as outras esperam.

A célula abaixo já vem pronta: a leitura do vídeo 4.1 e a função `linhas_por_particao` do vídeo 4.2.

In [ ]:
# vídeo 4.1: o feed e as partidas
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

eventos = spark.read.parquet(EVENTOS).withColumnRenamed("MatchID", "match_id")

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)

# vídeo 4.2: linhas por partição
def linhas_por_particao(df):
    """Quantas linhas ficaram em cada partição do DataFrame."""
    contagem = df.groupBy(F.spark_partition_id().alias("particao")).count().orderBy("particao")
    return [linha["count"] for linha in contagem.collect()]

## Primeiro passo: olhar a distribuição das chaves

Passes: cerca de 61% dos lances.

## Quando o skew atrapalha

- **agregação**: quase não sente (o `partial_count` resume antes do shuffle, vídeo 4.2);
- **join** pela chave desequilibrada: as linhas inteiras viajam. É o nosso caso.

## Reproduzindo o problema

Desligando **de propósito** o AQE e o broadcast automático, com 8 partições de shuffle:

Cinco números para 8 partições: três ficaram vazias. E uma tem mais de 1 milhão de linhas (os passes): **uma task
só**.

## Vendo na Spark UI

1. **Jobs**: o job mais recente.
2. **Stages**: o stage do join (8 tasks). Em **Summary Metrics**, compare **Median** e **Max** em **Shuffle Read Size /
   Records**.
3. A tabela de tasks: a task dos passes se destaca.
4. **SQL / DataFrame**: o plano desenhado, com as linhas de cada etapa.

## A solução: broadcast

7 linhas copiadas para cada executor: o feed não é embaralhado. O Spark não estima o tamanho de uma tabela criada de
uma lista Python, então indicamos com `F.broadcast()`:

As partições são as da leitura, equilibradas: sem shuffle, sem skew.

## E quando os dois lados são grandes?

O **AQE** (ligado por padrão) divide sozinho uma partição de join grande demais: mais de **5 vezes a mediana** e mais
de **256 MB**.

## O roteiro de diagnóstico

1. `explain()`: onde estão os `Exchange`?
2. Spark UI, **Stages**: muitas tasks minúsculas? Partições (ou AQE).
3. Uma task muito maior (**Max** contra **Median**)? `groupBy(chave).count()`.
4. Chave dominante `null` ou sem sentido? **Filtre**.
5. Dado legítimo? Lado pequeno: **broadcast**; dois lados grandes: **AQE**.

## Demanda da aula 4 atendida

| Etapa | Entrega |
|---|---|
| Task 1 | o plano do relatório lido e o tempo de antes medido |
| Task 2 | os três `Exchange` entendidos e o efeito das 200 partições medido |
| Task 3 | broadcast e AQE de volta, e o antes e o depois em `docs/otimizacao.md` |
| Task 4 | o desequilíbrio diagnosticado na Spark UI e resolvido com broadcast |

## Para consultar depois

- **Salting**, o último recurso para um join desequilibrado entre duas tabelas grandes: acrescentar ao lado grande
  uma coluna com um número aleatório de 0 a N-1 (o "sal"), replicar cada linha do outro lado N vezes, uma para cada
  valor de sal, e fazer o join por chave e sal. A chave dominante se espalha por N partições.
- **Skew em janelas:** uma window com `partitionBy` pela chave desequilibrada (aula 3) também leva todas as linhas
  dela para a mesma task.
- **Spill** na Spark UI: quando uma task não cabe na memória e grava dado temporário no disco. Spill concentrado numa
  task é outro sinal de partição grande demais.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()